In [ ]:
import os

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier
from lightgbm import LGBMClassifier

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import classification_report

import joblib

In [ ]:
data_path = 'G:\\HSUHK\\COM6003\\project\\archive'
meta_df = pd.read_csv(data_path + '\\metadata.csv')  # original dataset
img_train_2 = pd.read_csv(data_path+'\\meta_input_2.csv')
img_train_6 = pd.read_csv(data_path+'\\meta_input_6.csv')
img_val_2 = pd.read_csv(data_path+'\\meta_val_2.csv')
img_val_6 = pd.read_csv(data_path+'\\meta_val_6.csv')
img_test_2 = pd.read_csv(data_path+'\\meta_test_2.csv')
img_test_6 = pd.read_csv(data_path+'\\meta_test_6.csv')
img_index_2 = pd.read_csv(data_path+'\\imageIndex2.csv')
img_index_6 = pd.read_csv(data_path+'\\imageIndex6.csv')

# Metadata Processing

In [ ]:
meta_df.drop(columns=['biopsed','patient_id','lesion_id'],inplace=True)
meta_df.replace(['UNK','NaN',np.nan], 'UK',inplace=True)

In [ ]:
meta_df.replace('UK', np.nan,inplace=True)

In [ ]:
img_index_6.drop(columns='index',inplace=True)
img_index_2.drop(columns='index',inplace=True)

In [ ]:
x_train_2 = pd.concat([img_train_2,img_val_2],axis=0,ignore_index=True)
x_train_6 = pd.concat([img_train_6,img_val_6],axis=0,ignore_index=True)
x_test_2 = img_test_2
x_test_6 = img_test_6

In [ ]:
x_train_2.drop(columns=['Unnamed: 0'],inplace=True)
x_train_6.drop(columns=['Unnamed: 0'],inplace=True)
x_test_2.drop(columns=['Unnamed: 0'],inplace=True)
x_test_6.drop(columns=['Unnamed: 0'],inplace=True)

In [ ]:
x_train_2['img_id'] = x_train_2['0'].str.extract(r'([^\\]+\.png$)',expand=False)
x_train_2.drop(columns=['0'],inplace=True)

In [ ]:
x_train_6['img_id'] = x_train_6['0'].str.extract(r'([^\\]+\.png$)',expand=False)
x_train_6.drop(columns=['0'],inplace=True)

In [ ]:
x_test_2['img_id'] = x_test_2['0'].str.extract(r'([^\\]+\.png$)',expand=False)
x_test_2.drop(columns=['0'],inplace=True)

In [ ]:
x_test_6['img_id'] = x_test_6['0'].str.extract(r'([^\\]+\.png$)',expand=False)
x_test_6.drop(columns=['0'],inplace=True)

In [ ]:
y_train_2 = img_index_2[img_index_2['img_id'].isin(x_train_2['img_id'])]
y_train_6 = img_index_6[img_index_6['img_id'].isin(x_train_6['img_id'])]
y_test_2 = img_index_2[img_index_2['img_id'].isin(x_test_2['img_id'])]
y_test_6 = img_index_6[img_index_6['img_id'].isin(x_test_6['img_id'])]

In [ ]:
df_train_2 = pd.merge(x_train_2,meta_df,on='img_id',how='inner')
df_train_6 = pd.merge(x_train_6,meta_df,on='img_id',how='inner')
df_test_2 = pd.merge(x_test_2,meta_df,on='img_id',how='inner')
df_test_6 = pd.merge(x_test_6,meta_df,on='img_id',how='inner')

In [ ]:
df_train_2.drop(columns='diagnostic',inplace=True)
df_train_6.drop(columns='diagnostic',inplace=True)
df_test_2.drop(columns='diagnostic',inplace=True)
df_test_6.drop(columns='diagnostic',inplace=True)

In [ ]:
df_train_2.replace('UK',np.nan,inplace=True)
df_train_6.replace('UK',np.nan,inplace=True)
df_test_2.replace('UK',np.nan,inplace=True)
df_test_6.replace('UK',np.nan,inplace=True)

In [ ]:
def preprocess_data(df):
    imputer = IterativeImputer()
    enc = LabelEncoder()
    missing_percentage = round(df.isna().sum()*100/df.shape[0],1)
    for col, percentage in missing_percentage.items():
        if df[col].dtype == 'object':
            if percentage < 10:
                df[col] = enc.fit_transform(df[col])
                df[col] = imputer.fit_transform(df[[col]])
            else:
                df[col] = df[col].fillna('Unknown')
                df[col] = df[col].astype('str')
                df[col] = enc.fit_transform(df[col])
                df[col] = df[col].replace(len(enc.classes_), -1).astype('int64')

                # output the encode method
                encoding_mapping = dict(zip(enc.classes_, range(len(enc.classes_))))
                print(f"Feature '{col}':")
                for category, encoding in encoding_mapping.items():
                    if category == 'Unknown':
                        print(f"{category}: -1")
                    else:
                        print(f"{category}: {encoding}")
                print('-' * 30)
        else:
            if percentage < 10:
                df[col] = imputer.fit_transform(df[[col]])
            else:
                df[col] = df[col].fillna(-1)
                df[col] = df[col].astype('int64')

    return df

In [ ]:
df_train_2 = preprocess_data(df_train_2)
df_train_6 = preprocess_data(df_train_6)
df_test_2 = preprocess_data(df_test_2)
df_test_6 = preprocess_data(df_test_6)

In [ ]:
scaler = MinMaxScaler()
df_train_2 = scaler.fit_transform(df_train_2)
df_train_6 = scaler.fit_transform(df_train_6)
df_test_2 = scaler.fit_transform(df_test_2)
df_test_6 = scaler.fit_transform(df_test_6)

In [ ]:
y_train_2.drop(columns='img_id',inplace=True)
y_train_2.values.ravel()
y_train_6.drop(columns='img_id',inplace=True)
y_train_6.values.ravel()
y_test_2.drop(columns='img_id',inplace=True)
y_test_2.values.ravel()
y_test_6.drop(columns='img_id',inplace=True)
y_test_6.values.ravel()

In [ ]:
def textModel(x_train, y_train, x_test, y_test, class_weights=None):
    svm = SVC(kernel='linear', random_state=42)
    svm.fit(x_train, y_train)
    y_pred = svm.predict(x_test)
    print(f'Model 1: {classification_report(y_test, y_pred)}')
    
    rf = RandomForestClassifier(n_estimators=100, class_weight=class_weights, random_state=42)
    rf.fit(x_train, y_train)
    y_pred = rf.predict(x_test)
    print(f'Model 2: {classification_report(y_test, y_pred)}')

    xgb = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42)
    xgb.fit(x_train, y_train)
    y_pred = xgb.predict(x_test)
    print(f'Model 3: {classification_report(y_test, y_pred)}')

    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(x_train, y_train)
    y_pred = knn.predict(x_test)
    print(f'Model 4: {classification_report(y_test, y_pred)}')

    num_class = len(np.unique(y_train))
    if num_class == 2:
        num_class = 1

    lgbm = LGBMClassifier(num_class=num_class, class_weight=class_weights, random_state=42, verbose=-1)
    lgbm.fit(x_train, y_train)
    y_pred = lgbm.predict(x_test)
    print(f'Model 5: {classification_report(y_test, y_pred)}')

    return svm, rf, xgb, knn, lgbm

In [ ]:
svm1, rf1, xgb1, knn1, lgbm1 = textModel(df_train_2,y_train_2,df_test_2,y_test_2)

In [ ]:
svm2, rf2, xgb2, knn2, lgbm2 = textModel(df_train_6,y_train_6,df_test_6,y_test_6)

In [ ]:
estimators1 = [('svc', svm1), ('rf', rf1), ('xgb', xgb1), ('knn', knn1), ('lgbm', lgbm1)]
estimators2 = [('svc', svm2), ('rf', rf2), ('xgb', xgb2), ('knn', knn2), ('lgbm', lgbm2)]
stack_model1 = StackingClassifier(estimators=estimators1, final_estimator=XGBClassifier())
stack_model2 = StackingClassifier(estimators=estimators2, final_estimator=XGBClassifier())

In [ ]:
# fit the model on the training data
stack_model1.fit(df_train_2, y_train_2)
# make predictions
y_test_2_pred = stack_model1.predict(df_test_2)
# calculate the classification report
stack_model1_result = classification_report(y_test_2, y_test_2_pred)
print(f'Stacking Method for 2 Class Classification Result:\n{stack_model1_result}')

In [ ]:
stack_model2.fit(df_train_6, y_train_6)
y_test_6_pred = stack_model2.predict(df_test_6)
stack_model2_result = classification_report(y_test_6, y_test_6_pred)
print(f'Stacking Method for 6 Class Classification Result:\n{stack_model2_result}')

In [ ]:
joblib.dump(stack_model1_result, 'G:\\HSUHK\\COM6003\\project\\archive\\model_resultfinal_stack_2.joblib')
joblib.dump(stack_model2_result, 'G:\\HSUHK\\COM6003\\project\\archive\\model_resultfinal_stack_6.joblib')